# Stage 1: Data Preprocessing

This notebook loads the raw claim and evidence JSON files and produces three cleaned views of every text, each tailored to a different downstream consumer:

- **`bert`** — light cleaning (only unicode, whitespace and quotation normalisation). Used by BERT/RoBERTa-based encoders and the classifier, whose sub-word tokenisers expect natural text.
- **`bm25`** — aggressive cleaning (lowercased, punctuation removed, scientific abbreviations expanded so that for example `CO2` is indexed under both `co2` and `carbon dioxide`). Used to build the sparse BM25 index in Stage 2.
- **`keyword`** — lowercase, alphanumeric-only. Used by the keyword-overlap signal that complements BM25 in Stage 2.

Outputs are written to `./processed/` and consumed by every later stage.

## 1. Setup

Import standard libraries for JSON I/O, regex handling, unicode normalisation (`unidecode`), and progress bars (`tqdm`). The `OUTPUT_DIR` is created if it does not already exist.

In [ ]:
import sys

sys.path.insert(0, "../src")  # repo-local package
from collections import Counter

import numpy as np
from factcheck import config as cfg
from factcheck.config import load_json, save_json
from factcheck.text import clean_for_bert, clean_for_bm25, clean_for_keywords
from tqdm.auto import tqdm

In [ ]:
DATA_DIR = cfg.DATA_DIR
OUTPUT_DIR = cfg.PROC_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EVIDENCE_PATH = DATA_DIR / "evidence.json"
TRAIN_CLAIMS_PATH = DATA_DIR / "train-claims.json"
DEV_CLAIMS_PATH = DATA_DIR / "dev-claims.json"
TEST_CLAIMS_PATH = DATA_DIR / "test-claims-unlabelled.json"

## 2. Load raw JSON files

Load the four data files: the evidence corpus of about 1.2 million passages, the train claims, the dev claims, and the unlabelled test claims. The evidence file is the largest and takes roughly 30 seconds to read from disk.

In [ ]:
print("Loading evidence corpus (~30s for 1.2M passages)...")
evidence_raw = load_json(EVIDENCE_PATH)
train_claims = load_json(TRAIN_CLAIMS_PATH)
dev_claims = load_json(DEV_CLAIMS_PATH)
test_claims = load_json(TEST_CLAIMS_PATH)

print(f"  evidence    : {len(evidence_raw):,}")
print(f"  train claims: {len(train_claims):,}")
print(f"  dev claims  : {len(dev_claims):,}")
print(f"  test claims : {len(test_claims):,}")

In [ ]:
print("Train label distribution:")
for lbl, n in Counter(c["claim_label"] for c in train_claims.values()).most_common():
    print(f"  {lbl:<20s} {n}")

ev_per_claim = [len(c["evidences"]) for c in train_claims.values()]
print(f"avg evidences/claim: {np.mean(ev_per_claim):.2f}   min/max: {min(ev_per_claim)}/{max(ev_per_claim)}")
print(f"# claims with k gold evidences: {Counter(ev_per_claim).most_common()}")

## 3. Text cleaning

Defined in `src/factcheck/text.py`:
- `clean_for_bert` — unicode, quote and repeated-punctuation normalisation only.
- `clean_for_bm25` — additionally lowercases, strips punctuation and expands domain abbreviations while keeping both forms (`CO2 → co2 carbon dioxide`).
- `clean_for_keywords` — lowercase alphanumeric, for keyword overlap.


In [ ]:
# Quick sanity
demo = "The IPCC says CO2 levels rose ~50% and temperatures rose 1.5°C!!!"
print("RAW    :", demo)
print("BERT   :", clean_for_bert(demo))
print("BM25   :", clean_for_bm25(demo))
print("KEYWORD:", clean_for_keywords(demo))

## 4. Apply cleaning to every split

Run the three cleaning functions over (a) every evidence passage and (b) every claim in train / dev / test. Each evidence gets three keyed dictionaries (`evidence_bert`, `evidence_bm25`, `evidence_kw`) and each claim entry is augmented with `claim_bert`, `claim_bm25`, and `claim_keyword` fields. After that, simple length statistics on the BERT view are printed: these inform the truncation lengths chosen later for the encoders.

In [ ]:
def preprocess_evidence(evidence_dict):
    bert, bm25, kw = {}, {}, {}
    for eid, txt in tqdm(evidence_dict.items(), desc="evidence"):
        bert[eid] = clean_for_bert(txt)
        bm25[eid] = clean_for_bm25(txt)
        kw[eid] = clean_for_keywords(txt)
    return bert, bm25, kw


def preprocess_claims(claims_dict):
    out = {}
    for cid, entry in claims_dict.items():
        new = dict(entry)
        new["claim_bert"] = clean_for_bert(entry["claim_text"])
        new["claim_bm25"] = clean_for_bm25(entry["claim_text"])
        new["claim_keyword"] = clean_for_keywords(entry["claim_text"])
        out[cid] = new
    return out


evidence_bert, evidence_bm25, evidence_kw = preprocess_evidence(evidence_raw)
train_proc = preprocess_claims(train_claims)
dev_proc = preprocess_claims(dev_claims)
test_proc = preprocess_claims(test_claims)
print("done.")

In [ ]:
# Length statistics — useful for choosing truncation later
claim_lens = [len(c["claim_bert"].split()) for c in train_proc.values()]
ev_lens = [len(t.split()) for t in evidence_bert.values()]
for name, arr in [("claim", claim_lens), ("evidence", ev_lens)]:
    a = np.array(arr)
    print(
        f"{name:<10s} mean={a.mean():.1f}  median={np.median(a):.1f}  "
        f"p95={np.percentile(a, 95):.1f}  p99={np.percentile(a, 99):.1f}  max={a.max()}"
    )

## 5. Save processed files

Serialise each cleaned dictionary to JSON in `./processed/`. The evidence views are about 170 MB each. These files are the only input the later stages need from this notebook.

In [ ]:
save_json(evidence_bert, OUTPUT_DIR / "evidence_bert.json")
save_json(evidence_bm25, OUTPUT_DIR / "evidence_bm25.json")
save_json(evidence_kw, OUTPUT_DIR / "evidence_keyword.json")
save_json(train_proc, OUTPUT_DIR / "train_claims_proc.json")
save_json(dev_proc, OUTPUT_DIR / "dev_claims_proc.json")
save_json(test_proc, OUTPUT_DIR / "test_claims_proc.json")

## Summary

The notebook produces six files in `./processed/`:

- `evidence_bert.json`, `evidence_bm25.json`, `evidence_keyword.json` — three cleaned views of the full corpus.
- `train_claims_proc.json`, `dev_claims_proc.json`, `test_claims_proc.json` — each claim with its three cleaned fields added.

Length statistics on the BERT view show claims are short (median 19 tokens, p99 ≈ 48) and evidence passages are mostly short too (median 18 tokens, p99 ≈ 53), with a long tail up to ~480 tokens — useful when choosing truncation lengths for the encoders in Stages 3 and 4.